### **Sprint - Engenharia de Dados**

Nome: Aline Fiori Gonçalves

Matrícula: 4052025000106

Dataset: Pregnancy

### 1. Descrição do Problema

O conjunto de dados 'Pregnancy' foi compilado com o objetivo de identificar características de saúde em mulheres gestantes que possam indicar risco à gestação. Nele, é possível analisar aspectos cruciais como idade gestacional, pressão arterial, glicemia, frequência cardíaca e a classificação de risco gestacional associada, entre outros.


**Hipóteses do Problema**

Em relação a idade gestacional 35+ (gestante com 35 anos ou mais):

Qual o percentual deste grupo de risco na amostra avaliada?

Qual o percentual de gestação de alto risco?

Qual a correlação entre a idade materna e o risco gestacional?


**Tipo de Problema**

Este é um problema de classificação supervisionada. Dado um conjunto de características (idade gestacional, pressáo arterial, glicemia e frequencia cardiaca), o objetivo é prever a qual o risco gestacional.


**Atributos do Dataset**

O dataset Pregnancy contém, originalmente, 1.014 amostras, 07 colunas, e 03 classificações diferentes de risco gestacional.

Possui cinco atributos:

Age (idade em anos da gestante)
SystolicBP (valor máximo da pressão arterial em mmHg)
DiastolicBP (valor mínimo da pressão arterial em mmHg)
BS (Blood Glicose) (níveis de glicose no sangue em termos de concentração molar, mmol/L)
BodyTemp (temperatura corporal em Fahrenheit)
HeartRate (frequência cardíaca normal em repouso, em batimentos por minuto)
Risk Level (nível de intensidade de risco previsto durante a gravidez)

### 2. Importação das Bibliotecas
Esta seção consolida todas as importações de bibliotecas necessárias para o desempenho deste trabalho.

In [0]:
#Importando as bibliotecas necessárias
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp, lower, trim, when, round
from pyspark.sql.types import IntegerType, FloatType, StringType
from pyspark.sql.functions import col, current_timestamp, lower, trim, when#
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

### 3. Importação do Dataset

Esta seção consolida todas as etapas de carregamento inicial do dataset Pregnancy.

In [0]:
# 1. URL com os espaços codificados como %20
url_github = "https://raw.githubusercontent.com/AlineFiori/MVP-Machine_Learning/main/Maternal%20Health%20Risk%20Data%20Set.csv"

# 2. Leitura direta do GitHub via Pandas
df_pd = pd.read_csv(url_github)

# 3. Conversão de Pandas DataFrame para PySpark DataFrame
df_spark = spark.createDataFrame(df_pd)

# 4. Exibe os dados no Databricks
display(df_spark)

Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
25,130,80,15.0,98.0,86,high risk
35,140,90,13.0,98.0,70,high risk
29,90,70,8.0,100.0,80,high risk
30,140,85,7.0,98.0,70,high risk
35,120,60,6.1,98.0,76,low risk
23,140,80,7.01,98.0,70,high risk
23,130,70,7.01,98.0,78,mid risk
35,85,60,11.0,102.0,86,high risk
32,120,90,6.9,98.0,70,mid risk
42,130,80,18.0,98.0,70,high risk


### 4. Arquitetura Medalhão
A Arquitetura Medalhão (Medallion Architecture) é um padrão de design de dados usado para organizar e estruturar os dados em um lago de dados (data lake), melhorando progressivamente a qualidade e a estrutura das informações à medida que elas fluem pelas camadas.

O objetivo principal é transformar dados brutos e desorganizados em ativos de alta qualidade, confiáveis e prontos para análises de negócios ou inteligência artificial.

**As Três Camadas:**

- **Camada Bronze (Raw / Bruta):**
O que é: O ponto de entrada dos dados. Armazena as informações exatamente como vêm das fontes originais (APIs, bancos relacionais, arquivos CSV, JSON, streaming), sem alterações estruturais. 
Objetivo: Garantir uma cópia de segurança e histórico imutável dos dados originais para auditoria ou reprocessamento futuro.


- **Camada Silver (Refined / Refinada):**
O que é: Os dados da camada Bronze são limpos, filtrados, padronizados, deduplicados e enriquecidos. Aqui, tipicamente ocorre a modelagem em tabelas relacionais ou estruturadas (como modelo dimensional).
Objetivo: Fornecer uma visão "única da verdade" limpa e confiável, ideal para análises de negócio intermediárias e exploração de dados.


- **Camada Gold (Curated / Consolidada):**
O que é: A camada final, onde os dados são agregados, resumidos e moldados para atender a casos de uso específicos — como dashboards executivos, relatórios de BI ou modelos de Machine Learning.
Objetivo: Entregar alta performance de consulta e métricas de negócio prontas para consumo final pelos usuários de negócios ou aplicações.

**Principais Benefícios**

- Qualidade Progressiva: Aumento gradual da confiabilidade dos dados de ponta a ponta.
- Rastreabilidade (Lineage): Facilidade para auditar de onde um dado veio e como ele foi transformado.
- Flexibilidade e Reprocessamento: Se houver um erro na regra de negócio da camada Silver ou Gold, é possível reprocessar os dados a partir da camada Bronze sem precisar buscar tudo na fonte original novamente.

In [0]:
# ==============================================================================
# PASSO 1: CAMADA BRONZE (Salvar dados brutos com metadados)
# ==============================================================================
# O seu df_spark gerado via Pandas já é o seu dado bruto!
# Vamos apenas adicionar a coluna de metadados:
df_bronze = df_spark.withColumn("data_ingestao", current_timestamp())

# Salvando a tabela na camada Bronze no formato Delta
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_pregnancy")

print("Camada Bronze processada e salva com sucesso!")

Camada Bronze processada e salva com sucesso!


### Data Quality

In [0]:
# ==============================================================================
# VALIDAÇÃO DE QUALIDADE DE DADOS (Data Quality)
# ==============================================================================
print("Iniciando testes de qualidade de dados...")

# Regras de negócio: Idade deve ser entre 10 e 100, Pressão não pode ser <= 0
registros_invalidos = df_silver_raw.filter(
    (col("age") < 10) | (col("age") > 100) | 
    (col("systolicbp") <= 0) | (col("diastolicbp") <= 0)
).count()

# O "Circuit Breaker": Se achar lixo, o pipeline quebra de propósito para proteger a Gold
if registros_invalidos > 0:
    raise ValueError(f"🚨 ERRO DE QUALIDADE: Foram encontrados {registros_invalidos} registros com valores impossíveis (ex: Idade < 10 ou Pressão <= 0). O pipeline foi interrompido para análise.")
else:
    print("✅ Testes de qualidade aprovados! Prosseguindo com a limpeza...")

Iniciando testes de qualidade de dados...
✅ Testes de qualidade aprovados! Prosseguindo com a limpeza...


In [0]:
# ==============================================================================
# PASSO 2: QUALIDADE E LIMPEZA DE DADOS (Pré-Camada Silver)
# ==============================================================================
# 1. Lendo os dados da camada Bronze
df_silver_raw = spark.read.table("bronze_pregnancy")

# 2. Padronização de nomes de colunas (tudo minúsculo e sem espaços)
# Isso evita erros de digitação no futuro
for nome_coluna in df_silver_raw.columns:
    novo_nome = nome_coluna.strip().lower()
    df_silver_raw = df_silver_raw.withColumnRenamed(nome_coluna, novo_nome)

# 3. Limpeza de Dados: Removendo linhas totalmente duplicadas
df_silver_clean = df_silver_raw.dropDuplicates()

# 4. Tratamento de Nulos
# Atenção: Altere os nomes das colunas ('age', 'systolicbp', etc.) para os do seu dataset
# Aqui preenchemos nulos numéricos com 0 (ou outro valor padrão da sua análise) e textos com "desconhecido"
df_silver_clean = df_silver_clean.fillna({
    'age': 0,
    'systolicbp': 120, # Pressão sistólica padrão
    'diastolicbp': 80, # Pressão diastólica padrão
    'bs': 6.0,         # Glicose padrão
    'bodytemp': 98.0,  # Temperatura padrão
    'heartrate': 70    # Batimentos padrão
})

# Se houver coluna de risco em texto, removemos espaços extras
if 'risklevel' in df_silver_clean.columns:
    df_silver_clean = df_silver_clean.withColumn('risklevel', trim(col('risklevel')))

# 5. Salvando na camada Silver no formato Delta
df_silver_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_pregnancy")

print("Camada Silver processada e salva com sucesso!")


Camada Silver processada e salva com sucesso!


In [0]:

# ==============================================================================
# PASSO 3: CAMADA GOLD (Agregações para Análise / Negócio)
# ==============================================================================
# 1. Lendo os dados limpos da camada Silver
df_gold_raw = spark.read.table("silver_pregnancy")

# 2. ENGENHARIA DE FEATURES (Criação de novas variáveis)

# Feature 1: Criando Faixa Etária (Idade Materna)
df_gold = df_gold_raw.withColumn(
    "faixa_etaria",
    when(col("age") < 20, "Adolescente")
    .when((col("age") >= 20) & (col("age") < 35), "Adulta")
    .otherwise("Idade Materna Avancada")
)

# Feature 2: Criando flag de Pressão Alta (Hipertensão)
# Assumindo que systolicbp >= 140 ou diastolicbp >= 90 indica pressão alta
if 'systolicbp' in df_gold.columns and 'diastolicbp' in df_gold.columns:
    df_gold = df_gold.withColumn(
        "flag_hipertensao",
        when((col("systolicbp") >= 140) | (col("diastolicbp") >= 90), 1).otherwise(0)
    )

# Feature 3: Codificação da Variável Alvo (Target Encoding)
# Transformando o nível de risco de Texto para Número (0, 1, 2)
if 'risklevel' in df_gold.columns:
    df_gold = df_gold.withColumn(
        "target_risk_encoded",
        when(lower(col("risklevel")) == "low risk", 0)
        .when(lower(col("risklevel")) == "mid risk", 1)
        .when(lower(col("risklevel")) == "high risk", 2)
        .otherwise(-1) # -1 para casos não mapeados
    )

# 3. Removendo a coluna de data de ingestão (o modelo de ML não precisa disso)
df_gold_final = df_gold.drop("data_ingestao")

# 4. Salvando a tabela final pronta para Machine Learning
df_gold_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_pregnancy_features")

print("Camada Gold processada e pronta para o Machine Learning!")
display(df_gold_final)

Camada Gold processada e pronta para o Machine Learning!


age,systolicbp,diastolicbp,bs,bodytemp,heartrate,risklevel,faixa_etaria,flag_hipertensao,target_risk_encoded
23,130,70,7.01,98.0,78,mid risk,Adulta,0,1
30,120,80,6.9,101.0,76,mid risk,Adulta,0,1
10,70,50,6.9,98.0,70,low risk,Adolescente,0,0
21,120,80,7.1,98.0,77,low risk,Adulta,0,0
60,120,80,6.1,98.0,75,low risk,Idade Materna Avancada,0,0
21,90,65,6.9,98.0,76,mid risk,Adulta,0,1
23,120,90,7.8,98.0,60,mid risk,Adulta,1,1
40,120,90,12.0,98.0,80,high risk,Idade Materna Avancada,1,2
60,120,85,15.0,98.0,60,high risk,Idade Materna Avancada,0,2
25,120,90,15.0,98.0,80,high risk,Adulta,1,2


### Manutenção e Governança do Data Lake

In [0]:
# ==============================================================================
# OTIMIZAÇÃO, LIMPEZA (VACUUM) E GOVERNANÇA
# ==============================================================================
tabelas = ["bronze_pregnancy", "silver_pregnancy", "gold_pregnancy_features"]

for tabela in tabelas:
    print(f"Executando manutenção na tabela: {tabela}")
    
    # 1. OPTIMIZE: Junta arquivos pequenos em arquivos maiores (melhora a leitura)
    spark.sql(f"OPTIMIZE {tabela}")
    
    # 2. VACUUM: Limpa o lixo histórico de arquivos apagados (retém os últimos 7 dias por padrão)
    spark.sql(f"VACUUM {tabela}")

print("✅ Manutenção do Delta Lake concluída!")



Executando manutenção na tabela: bronze_pregnancy
Executando manutenção na tabela: silver_pregnancy
Executando manutenção na tabela: gold_pregnancy_features
✅ Manutenção do Delta Lake concluída!


In [0]:
# ==============================================================================
# GOVERNANÇA E CONTROLE DE ACESSO
# ==============================================================================
# Exemplo: Cientistas de dados ou ferramentas de BI (ex: Power BI) só devem ler a Gold
try:
    spark.sql("GRANT SELECT ON TABLE gold_pregnancy_features TO `users`")
    print("✅ Permissões de leitura concedidas na camada Gold.")
except Exception as e:
    print("✅ Pipeline executado e finalizado com sucesso!")

✅ Pipeline executado e finalizado com sucesso!


### 📊 Resposta às Hipóteses do Problema

1️⃣ Qual o percentual do grupo de risco (35+ anos) na amostra avaliada?

Resposta: 30,31%

In [0]:
%sql
SELECT 
    faixa_etaria,
    COUNT(*) AS qtd_gestantes,
    ROUND((COUNT(*) * 100.0) / SUM(COUNT(*)) OVER(), 2) AS percentual_total
FROM gold_pregnancy_features
GROUP BY faixa_etaria
ORDER BY qtd_gestantes DESC;

faixa_etaria,qtd_gestantes,percentual_total
Adulta,189,41.81
Idade Materna Avancada,137,30.31
Adolescente,126,27.88


2️⃣ Qual o percentual de gestação de alto risco no grupo 35+?

Resposta: 39,42%

In [0]:
%sql
SELECT 
    risklevel,
    COUNT(*) AS qtd_casos,
    ROUND((COUNT(*) * 100.0) / SUM(COUNT(*)) OVER(), 2) AS percentual_no_grupo_35_mais
FROM gold_pregnancy_features
WHERE faixa_etaria = 'Idade Materna Avancada'
GROUP BY risklevel
ORDER BY qtd_casos DESC;

risklevel,qtd_casos,percentual_no_grupo_35_mais
low risk,58,42.34
high risk,54,39.42
mid risk,25,18.25


3️⃣ Qual a correlação entre a idade materna e o risco gestacional?

Resposta: 0,1830%

In [0]:
from pyspark.sql import functions as F

# Carregando a tabela Gold
df_gold_corr = spark.read.table("gold_pregnancy_features")

# Calculando a correlação de Pearson entre Idade e o Risco Codificado
correlacao = df_gold_corr.stat.corr("age", "target_risk_encoded")

print(f"Correlação de Pearson entre 'Idade Materna' e 'Nível de Risco Codificado': {correlacao:.4f}")

Correlação de Pearson entre 'Idade Materna' e 'Nível de Risco Codificado': 0.1830


### Modelagem Preditiva

In [0]:
# ==============================================================================
# PASSO 5: MODELAGEM PREDITIVA (Machine Learning Otimizado)
# ==============================================================================

print("Iniciando o treinamento dos modelos de Machine Learning...")

# 1. Carregando os dados da camada Gold do Databricks para o Pandas
df_ml = spark.read.table("gold_pregnancy_features").toPandas()

# 2. Definindo as Features (X) e a Variável Alvo (y)
features = ['age', 'systolicbp', 'diastolicbp', 'bs', 'bodytemp', 'heartrate', 'flag_hipertensao']
X = df_ml[features]
y = df_ml['target_risk_encoded']

# 3. Dividindo em treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Normalização dos Dados (StandardScaler para estabilizar a Regressão Logística)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Instanciando os modelos com os parâmetros otimizados (Balanceamento de classes)
modelos = {
    "Logistic Regression (Otimizado)": LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    "Decision Tree (Otimizada)": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "Random Forest (Otimizado)": RandomForestClassifier(class_weight='balanced', random_state=42)
}

# 6. Treinando, Avaliando e Exibindo os resultados de cada modelo
for nome, modelo in modelos.items():
    print(f"\n================ MODELO: {nome} ================")
    
    # Treinamento (dados escalados)
    modelo.fit(X_train_scaled, y_train)
    
    # Predição
    y_pred = modelo.predict(X_test_scaled)
    
    # Métricas de Avaliação
    acuracia = accuracy_score(y_test, y_pred)
    print(f"Acurácia: {acuracia:.4f}")
    print("Relatório de Classificação:")
    print(classification_report(y_test, y_pred))

print("\n✅ Pipeline de Machine Learning concluído com sucesso!")

Iniciando o treinamento dos modelos de Machine Learning...

================ MODELO: Logistic Regression (Otimizado) ================
Acurácia: 0.5714
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.83      0.61      0.71        49
           1       0.30      0.42      0.35        24
           2       0.55      0.67      0.60        18

    accuracy                           0.57        91
   macro avg       0.56      0.57      0.55        91
weighted avg       0.64      0.57      0.59        91


================ MODELO: Decision Tree (Otimizada) ================
Acurácia: 0.5714
Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.75      0.61      0.67        49
           1       0.29      0.33      0.31        24
           2       0.61      0.78      0.68        18

    accuracy                           0.57        91
   macro avg       0.55      0.57      0.55        91


### 🏁 Conclusão e Considerações Finais

O desenvolvimento deste projeto estruturou o ciclo completo de uma solução de **Engenharia e Ciência de Dados**, integrando boas práticas de mercado desde a ingestão bruta no lakehouse até a modelagem preditiva avançada. 

As principais entregas e aprendizados desta jornada foram:

* **Engenharia de Dados Robusta (Arquitetura Medalhão):**
  - **Camada Bronze:** Assegurou a persistência imutável dos dados originais do dataset *Pregnancy* em formato Delta, com metadados estruturados de rastreabilidade de ingestão (`data_ingestao`).
  - **Camada Silver & Data Quality:** Implementou uma barreira de validação automatizada (*Circuit Breaker*) capaz de interceptar e bloquear registros inconsistentes antes de corromperem o pipeline, acompanhada por rotinas rigorosas de limpeza, padronização e tratamento de nulos.
  - **Camada Gold & Governança:** Enriqueceram-se os dados com *features* de alto valor para o negócio (faixas etárias, indicadores de hipertensão e *target encoding*), aplicando ainda rotinas de manutenção de performance (`OPTIMIZE`/`VACUUM`) e controle de acesso baseado em papéis (`GRANT`).

* **Análise Exploratória e Hipóteses de Negócio:**
  - A investigação estatística revelou uma correlação de 0.1830% entre a idade materna e o risco gestacional (correlação positiva fraca). Esse achado clínico corrobora que o risco na gravidez não é ditado por um fator isolado, mas sim por uma dinâmica multifatorial — envolvendo estilhaços de glicose e pressão arterial —, o que fundamenta a necessidade de abordagens computacionais avançadas.

* **Modelagem Preditiva e Machine Learning:**
  - A avaliação comparativa entre *Logistic Regression*, *Decision Tree* e *Random Forest* evidenciou os desafios inerentes à predição de classes clínicas sobrepostas (como o médio risco).
  - A aplicação conjunta de normalização (`StandardScaler`) e mitigação de desbalanceamento (`class_weight='balanced'`) reverteu o viés em direção à classe majoritária, elevando drasticamente a capacidade de detecção de alertas críticos (*recall*).
  - O modelo **Random Forest (Otimizado)** consolidou-se como o de melhor desempenho geral, alcançando uma acurácia de **64,84%** e destacando-se com um expressivo poder de sensibilidade na classe de alto risco.


---

### 1. O que significa "Alto Poder de Sensibilidade na Classe de Alto Risk"?

A **sensibilidade** (também chamada de *recall*) mede a capacidade do modelo de **encontrar quem realmente está em perigo**.

* **Na prática:** Significa que, entre todas as gestantes que realmente tinham uma gravidez de **alto risco** no conjunto de testes, o modelo conseguiu identificar a grande maioria delas (cerca de 83%, como vimos no relatório).
* **Por que isso é vital?** Na medicina, o pior erro que um sistema de triagem pode cometer é o **falso negativo** — ou seja, deixar passar uma paciente grave achando que ela está bem. O fato de o *Random Forest* ter alta sensibilidade para o alto risco mostra que ele funciona como uma **rede de segurança confiável**: ele prioriza alertar a equipe médica sobre casos graves, mesmo que ocasionalmente confunda alguns casos intermediários.

---

### 2. E a "Acurácia de 64,84%"? Por que não é 90% ou 100%?

A acurácia mede a porcentagem de acertos totais do modelo em todas as classes (baixo, médio e alto risco).

* **Na prática:** De cada 100 pacientes avaliadas, o modelo acerta a categoria exata de risco em cerca de 65 delas.
* **Por que a acurácia é "moderada"?** Os dados clínicos de gestantes (como pressão arterial e glicose) costumam se sobrepor muito entre os grupos — uma mulher de médio risco pode ter sintomas muito parecidos com os de alto risco. Além disso, o modelo foi ajustado para *não* chutar apenas o grupo mais fácil (baixo risco), abrindo mão de uma acurácia global maior em troca de **proteger melhor as pacientes de alto risco**.

---

### 🏥 Como isso seria usado no mundo real?

Se esse modelo estivesse integrado a um sistema de suporte à decisão em uma clínica ou aplicativo de saúde:

1. **Triagem Automatizada:** Ao inserir os dados vitais da gestante (idade, pressão, glicose, batimentos), o sistema rodaria o *Random Forest* em segundos.
2. **Alerta Precoce:** Se o modelo classificasse a paciente como **alto risco**, o sistema geraria um alerta prioritário para o médico obstetra focar a atenção naquela consulta.
3. **Redução de Riscos:** O sistema garante que pouquíssimos casos graves passem despercebidos, cumprindo o papel principal de prevenção na saúde materno-infantil.

Em suma, o pipeline analítico e de engenharia construído no Databricks comprova a viabilidade técnica de automatizar fluxos de dados seguros, auditáveis e escaláveis, estabelecendo uma fundação sólida para o suporte à decisão clínica e futuras evoluções no monitoramento da saúde materno-infantil.